# 03 — Backlog Pressure Analysis

Identify continuous periods where transfers into HHS exceed discharges, then
rank elevated episodes by duration and cumulative pressure.

In [ ]:
from pathlib import Path
import sys
import numpy as np  # noqa: F401 -- shared setup; used by modeling notebooks
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
from app_utils import NET_INTAKE_COLUMN
from backend.analytics import calculate_backlog_episodes
from src.visualisation import VisualizationConfig, create_net_intake_backlog_chart

metrics = pd.read_csv(
    PROCESSED_DIR / "uac_capacity_metrics_daily.csv",
    parse_dates=["Date"],
).set_index("Date")
episodes = calculate_backlog_episodes(metrics, threshold_days=3)
elevated = episodes.loc[episodes["Elevated"]].copy()
print({
    "all_positive_pressure_episodes": len(episodes),
    "elevated_episodes": len(elevated),
    "longest_streak_days": int(episodes["Duration (Days)"].max()),
    "positive_pressure_days": int(metrics[NET_INTAKE_COLUMN].gt(0).sum()),
})
elevated.head(15)

In [ ]:
annual_pressure = metrics.groupby(metrics.index.year)[NET_INTAKE_COLUMN].agg(
    cumulative_net_intake="sum",
    average_daily_pressure="mean",
    positive_pressure_days=lambda values: int(values.gt(0).sum()),
    peak_daily_pressure="max",
)
annual_pressure

In [ ]:
backlog_figure = create_net_intake_backlog_chart(
    metrics,
    VisualizationConfig(granularity="Weekly", backlog_threshold_days=3),
)
backlog_figure

In [ ]:
export_path = OUTPUT_DIR / "exports" / "backlog_episodes.csv"
export_path.parent.mkdir(parents=True, exist_ok=True)
episodes.to_csv(export_path, index=False, date_format="%Y-%m-%d")
print(f"Wrote {export_path}")

A positive net-intake streak is an operational flow signal,
not a statement about care quality or an individual child's placement.